# Example 7.2 / 1: Fixed Costs and Activation

Companion notebook for **7.2 Binary Variables and Logical Conditions** in the AMPL lab Documentation, section "Fixed Costs and Activation". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Fixed Costs and Activation

In the steel model, suppose each product also has a fixed setup cost paid only if any of it is
made. Let `Use[p]` indicate whether production of `p` is active; the constraint `Activate` allows
`Make[p]` to be positive only when `Use[p] = 1`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                         # products
param rate {PROD} > 0;            # tons produced per hour
param profit {PROD};              # profit per ton
param market {PROD} >= 0;         # most that can be sold, in tons
param setup {PROD} >= 0;          # fixed cost paid if the product is made
param avail >= 0;                 # hours available

var Make {p in PROD} >= 0, <= market[p];  # tons produced
var Use {PROD} binary;                     # 1 if production of the product is active

# profit per ton minus setup costs of active products
maximize NetProfit:
    sum {p in PROD} (profit[p] * Make[p] - setup[p] * Use[p]);
# hours used fit within the hours available
subject to Time: sum {p in PROD} (1 / rate[p]) * Make[p] <= avail;
# production only where the setup is paid
subject to Activate {p in PROD}:
    Make[p] <= market[p] * Use[p];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"bands": 200, "coils": 140, "plate": 160}
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}
ampl.param["setup"] = {"bands": 20000, "coils": 10000, "plate": 50000}
ampl.param["avail"] = 40

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.get_data("Make", "Use").to_pandas())

HiGHS 1.14.0: optimal solution; objective 162000
1 simplex iterations
1 branching nodes
       Make  Use
bands  6000    1
coils  1400    1
plate     0    0


### Pure AMPL: data.dat and model.run

`ampl.get_data("Make", "Use")` returns both variables in one table. Bands fill 30 of the 40 hours.
The last 10 hours could make 1,600 tons of plate (46,400 profit) or 1,400 tons of coils (42,000),
but plate's setup cost of 50,000 exceeds what it would earn, so the model pays the 10,000 coils
setup instead and leaves `Use["plate"] = 0`.

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
param: PROD: rate profit market setup :=
  bands  200  25  6000  20000
  coils  140  30  4000  10000
  plate  160  29  3500  50000 ;
param avail := 40;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Make, Use;

HiGHS 1.14.0: optimal solution; objective 162000
1 simplex iterations
1 branching nodes
:       Make Use    :=
bands   6000   1
coils   1400   1
plate      0   0
;

